# Sensitivity Analysis for GPU-Accelerated OrdinalSustain

Comprehensive stress-testing with synthetic data to validate robustness:
1. Vary number of subtypes (1-6)
2. Test missing data rates (0%, 10%, 20%, 30%)
3. Determine minimum viable sample size
4. Test different noise levels (0-30%)
5. Combined stress test

**Runtime:** ~25-30 min on T4 GPU (all 5 experiments)

In [ ]:
# Cell 1: Check GPU and install dependencies
!nvidia-smi
!pip install torch numpy scipy matplotlib seaborn pandas tqdm scikit-learn tabulate -q

In [ ]:
# Cell 2: Clone repo and setup
import os
if not os.path.exists('/content/mphil'):
    !git clone https://github.com/Amelia3141/mphil.git /content/mphil
%cd /content/mphil
!git checkout claude/convert-to-jupyter-01GY8iZvAixjYs4t3VyLsWHf
!git pull origin claude/convert-to-jupyter-01GY8iZvAixjYs4t3VyLsWHf

import sys
sys.path.insert(0, '/content/mphil')

In [ ]:
# Cell 3: Imports and GPU check
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

from pySuStaIn.TorchOrdinalSustain import TorchOrdinalSustain

# Check GPU
if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {device} ({mem:.1f} GB)")
    USE_GPU = True
else:
    print("No GPU detected - running on CPU (will be slower)")
    USE_GPU = False

# Create output directories
for d in ['sensitivity_output/figures', 'sensitivity_output/tables', 'sensitivity_output/data']:
    Path(d).mkdir(parents=True, exist_ok=True)

# Plot style
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)
plt.rcParams['figure.dpi'] = 150
print('Setup complete. Output directories created.')

In [ ]:
# Cell 4: Synthetic data generator

def generate_synthetic_data(n_subjects=1000, n_biomarkers=13, n_scores=3,
                            n_subtypes=3, missing_rate=0.0, noise_level=0.0, seed=42):
    """Generate realistic synthetic ordinal data with known ground truth."""
    np.random.seed(seed)

    # Generate true event sequences for each subtype
    true_sequences = []
    for s in range(n_subtypes):
        sequence = np.arange(n_biomarkers * n_scores)
        np.random.shuffle(sequence)
        true_sequences.append(sequence)

    true_subtypes = np.random.choice(n_subtypes, n_subjects)
    max_stage = n_biomarkers * n_scores
    true_stages = np.random.randint(0, max_stage + 1, n_subjects)

    p_correct = 0.9 - noise_level
    p_nl_dist = np.full((n_scores + 1), (1 - p_correct) / n_scores)
    p_nl_dist[0] = p_correct

    p_score_dist = np.full((n_scores, n_scores + 1), (1 - p_correct) / n_scores)
    for score in range(n_scores):
        p_score_dist[score, score + 1] = p_correct

    data = np.zeros((n_subjects, n_biomarkers), dtype=int)
    for i in range(n_subjects):
        subtype = true_subtypes[i]
        stage = true_stages[i]
        sequence = true_sequences[subtype]
        for b in range(n_biomarkers):
            biomarker_events = sequence[b * n_scores:(b + 1) * n_scores]
            events_occurred = np.sum(biomarker_events < stage)
            data[i, b] = min(events_occurred, n_scores) if events_occurred > 0 else 0

    if noise_level > 0:
        n_flips = int(n_subjects * n_biomarkers * noise_level)
        flip_indices = np.random.choice(n_subjects * n_biomarkers, n_flips, replace=False)
        for idx in flip_indices:
            i, b = divmod(idx, n_biomarkers)
            current = data[i, b]
            if current > 0:
                data[i, b] = max(0, current + np.random.choice([-1, 1]))

    prob_nl = np.zeros((n_subjects, n_biomarkers))
    prob_score = np.zeros((n_subjects, n_biomarkers, n_scores))
    for i in range(n_subjects):
        for b in range(n_biomarkers):
            score = data[i, b]
            prob_nl[i, b] = p_nl_dist[score]
            for z in range(n_scores):
                prob_score[i, b, z] = p_score_dist[z, score]

    if missing_rate > 0:
        n_missing = int(n_subjects * n_biomarkers * missing_rate)
        missing_indices = np.random.choice(n_subjects * n_biomarkers, n_missing, replace=False)
        for idx in missing_indices:
            i, b = divmod(idx, n_biomarkers)
            prob_nl[i, b] = 1.0 / (n_scores + 1)
            prob_score[i, b, :] = 1.0 / (n_scores + 1)

    score_vals = np.tile(np.arange(1, n_scores + 1), (n_biomarkers, 1))
    biomarker_labels = [f'Biomarker_{i+1}' for i in range(n_biomarkers)]

    return prob_nl, prob_score, score_vals, biomarker_labels, true_subtypes, true_stages

print('Data generator ready.')

In [ ]:
# Cell 5: Single experiment runner

def run_experiment(n_subjects, n_biomarkers, n_scores, n_subtypes_true,
                   n_subtypes_fit, missing_rate, noise_level, seed,
                   use_gpu=True, device_id=0, output_base='./sensitivity_output/data'):
    """Run a single SuStaIn experiment and return metrics."""

    prob_nl, prob_score, score_vals, labels, true_subtypes, true_stages = \
        generate_synthetic_data(n_subjects, n_biomarkers, n_scores,
                                n_subtypes_true, missing_rate, noise_level, seed)

    output_folder = Path(output_base) / f'exp_n{n_subjects}_N{n_subtypes_fit}_m{int(missing_rate*100)}_noise{int(noise_level*100)}'
    output_folder.mkdir(parents=True, exist_ok=True)

    try:
        start_time = time.time()

        sustain = TorchOrdinalSustain(
            prob_nl, prob_score, score_vals, labels,
            N_startpoints=10,
            N_S_max=n_subtypes_fit,
            N_iterations_MCMC=5000,
            output_folder=str(output_folder),
            dataset_name='sensitivity',
            use_parallel_startpoints=False,
            seed=seed,
            use_gpu=use_gpu,
            device_id=device_id
        )

        samples_sequence, samples_f, ml_subtype, prob_ml_subtype, \
        ml_stage, prob_ml_stage, prob_subtype_stage = sustain.run_sustain_algorithm()

        runtime = time.time() - start_time

        # Squeeze from (n,1) to (n,)
        ml_subtype = np.squeeze(ml_subtype)
        ml_stage = np.squeeze(ml_stage)
        prob_ml_subtype = np.squeeze(prob_ml_subtype)
        prob_ml_stage = np.squeeze(prob_ml_stage)

        subtype_accuracy = np.mean(ml_subtype == true_subtypes) if n_subtypes_fit == n_subtypes_true else np.nan
        stage_mae = np.mean(np.abs(ml_stage - true_stages))
        stage_correlation = np.corrcoef(ml_stage, true_stages)[0, 1]
        mean_confidence = np.mean(prob_ml_subtype)

        return {
            'n_subjects': n_subjects, 'n_biomarkers': n_biomarkers, 'n_scores': n_scores,
            'n_subtypes_true': n_subtypes_true, 'n_subtypes_fit': n_subtypes_fit,
            'missing_rate': missing_rate, 'noise_level': noise_level,
            'runtime': runtime, 'subtype_accuracy': subtype_accuracy,
            'stage_mae': stage_mae, 'stage_correlation': stage_correlation,
            'mean_confidence': mean_confidence, 'success': True, 'error': None
        }

    except Exception as e:
        print(f'ERROR: {e}')
        return {
            'n_subjects': n_subjects, 'n_biomarkers': n_biomarkers, 'n_scores': n_scores,
            'n_subtypes_true': n_subtypes_true, 'n_subtypes_fit': n_subtypes_fit,
            'missing_rate': missing_rate, 'noise_level': noise_level,
            'runtime': np.nan, 'subtype_accuracy': np.nan,
            'stage_mae': np.nan, 'stage_correlation': np.nan,
            'mean_confidence': np.nan, 'success': False, 'error': str(e)
        }

print('Experiment runner ready.')

In [ ]:
# Cell 6: Experiment 1 - Vary number of subtypes (1-6)
print('='*60)
print('EXPERIMENT 1: Varying Number of Subtypes'.center(60))
print('='*60)

results_exp1 = []
for n_subtypes in range(1, 7):
    print(f'\nRunning N={n_subtypes} subtypes...')
    result = run_experiment(
        n_subjects=1000, n_biomarkers=13, n_scores=3,
        n_subtypes_true=n_subtypes, n_subtypes_fit=n_subtypes,
        missing_rate=0.0, noise_level=0.0, seed=42+n_subtypes,
        use_gpu=USE_GPU
    )
    results_exp1.append(result)
    if result['success']:
        print(f'  Runtime: {result["runtime"]:.1f}s | Correlation: {result["stage_correlation"]:.3f}')
    else:
        print(f'  FAILED: {result["error"]}')

df1 = pd.DataFrame(results_exp1)
df1.to_csv('./sensitivity_output/tables/experiment_1_vary_subtypes.csv', index=False)
print('\nExperiment 1 complete!')
df1[['n_subtypes_fit', 'runtime', 'stage_correlation', 'mean_confidence', 'success']]

In [ ]:
# Cell 7: Plot Experiment 1
df1_ok = df1[df1['success']]
if not df1_ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(df1_ok['n_subtypes_fit'], df1_ok['runtime'], 'o-', lw=2, ms=8)
    axes[0].set_xlabel('Number of Subtypes')
    axes[0].set_ylabel('Runtime (seconds)')
    axes[0].set_title('Runtime Scaling with Subtypes')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df1_ok['n_subtypes_fit'], df1_ok['stage_correlation'], 'o-', lw=2, ms=8, color='green')
    axes[1].set_xlabel('Number of Subtypes')
    axes[1].set_ylabel('Stage Correlation')
    axes[1].set_title('Stage Prediction Accuracy')
    axes[1].set_ylim([0, 1])
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('./sensitivity_output/figures/exp1_vary_subtypes.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No successful experiments to plot.')

In [ ]:
# Cell 8: Experiment 2 - Missing data (0%, 10%, 20%, 30%)
print('='*60)
print('EXPERIMENT 2: Missing Data Robustness'.center(60))
print('='*60)

results_exp2 = []
for missing_rate in [0.0, 0.1, 0.2, 0.3]:
    print(f'\nRunning with {missing_rate:.0%} missing data...')
    result = run_experiment(
        n_subjects=1000, n_biomarkers=13, n_scores=3,
        n_subtypes_true=3, n_subtypes_fit=3,
        missing_rate=missing_rate, noise_level=0.0, seed=42,
        use_gpu=USE_GPU
    )
    results_exp2.append(result)
    if result['success']:
        print(f'  Correlation: {result["stage_correlation"]:.3f} | MAE: {result["stage_mae"]:.2f}')

df2 = pd.DataFrame(results_exp2)
df2.to_csv('./sensitivity_output/tables/experiment_2_missing_data.csv', index=False)
print('\nExperiment 2 complete!')
df2[['missing_rate', 'stage_correlation', 'stage_mae', 'mean_confidence', 'success']]

In [ ]:
# Cell 9: Plot Experiment 2
df2_ok = df2[df2['success']]
if not df2_ok.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    missing_pct = df2_ok['missing_rate'] * 100

    axes[0].plot(missing_pct, df2_ok['stage_correlation'], 'o-', lw=2, ms=8, color='blue')
    axes[0].set_xlabel('Missing Data (%)')
    axes[0].set_ylabel('Stage Correlation')
    axes[0].set_title('Stage Prediction vs Missing Data')
    axes[0].set_ylim([0, 1])
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(missing_pct, df2_ok['stage_mae'], 'o-', lw=2, ms=8, color='red')
    axes[1].set_xlabel('Missing Data (%)')
    axes[1].set_ylabel('Stage MAE')
    axes[1].set_title('Stage Error vs Missing Data')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(missing_pct, df2_ok['mean_confidence'], 'o-', lw=2, ms=8, color='green')
    axes[2].set_xlabel('Missing Data (%)')
    axes[2].set_ylabel('Mean Confidence')
    axes[2].set_title('Subtype Confidence vs Missing Data')
    axes[2].set_ylim([0, 1])
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('./sensitivity_output/figures/exp2_missing_data.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 10: Experiment 3 - Minimum viable sample size
print('='*60)
print('EXPERIMENT 3: Minimum Viable Sample Size'.center(60))
print('='*60)

results_exp3 = []
for n_subjects in [100, 250, 500, 1000, 2000, 5000]:
    print(f'\nRunning with n={n_subjects}...')
    result = run_experiment(
        n_subjects=n_subjects, n_biomarkers=13, n_scores=3,
        n_subtypes_true=3, n_subtypes_fit=3,
        missing_rate=0.0, noise_level=0.0, seed=42,
        use_gpu=USE_GPU
    )
    results_exp3.append(result)
    if result['success']:
        print(f'  Runtime: {result["runtime"]:.1f}s | Correlation: {result["stage_correlation"]:.3f}')

df3 = pd.DataFrame(results_exp3)
df3.to_csv('./sensitivity_output/tables/experiment_3_sample_size.csv', index=False)
print('\nExperiment 3 complete!')
df3[['n_subjects', 'runtime', 'stage_correlation', 'mean_confidence', 'success']]

In [ ]:
# Cell 11: Plot Experiment 3
df3_ok = df3[df3['success']]
if not df3_ok.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].semilogx(df3_ok['n_subjects'], df3_ok['stage_correlation'], 'o-', lw=2, ms=8)
    axes[0].set_xlabel('Sample Size (n)')
    axes[0].set_ylabel('Stage Correlation')
    axes[0].set_title('Accuracy vs Sample Size')
    axes[0].set_ylim([0, 1])
    axes[0].axhline(y=0.8, color='r', ls='--', alpha=0.5, label='Target (0.8)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].loglog(df3_ok['n_subjects'], df3_ok['runtime'], 'o-', lw=2, ms=8, color='orange')
    axes[1].set_xlabel('Sample Size (n)')
    axes[1].set_ylabel('Runtime (seconds)')
    axes[1].set_title('Runtime Scaling')
    axes[1].grid(True, alpha=0.3)

    axes[2].semilogx(df3_ok['n_subjects'], df3_ok['mean_confidence'], 'o-', lw=2, ms=8, color='green')
    axes[2].set_xlabel('Sample Size (n)')
    axes[2].set_ylabel('Mean Confidence')
    axes[2].set_title('Confidence vs Sample Size')
    axes[2].set_ylim([0, 1])
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('./sensitivity_output/figures/exp3_sample_size.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 12: Experiment 4 - Noise levels
print('='*60)
print('EXPERIMENT 4: Noise Robustness'.center(60))
print('='*60)

results_exp4 = []
for noise_level in [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]:
    print(f'\nRunning with {noise_level:.0%} noise...')
    result = run_experiment(
        n_subjects=1000, n_biomarkers=13, n_scores=3,
        n_subtypes_true=3, n_subtypes_fit=3,
        missing_rate=0.0, noise_level=noise_level, seed=42,
        use_gpu=USE_GPU
    )
    results_exp4.append(result)
    if result['success']:
        print(f'  Correlation: {result["stage_correlation"]:.3f} | MAE: {result["stage_mae"]:.2f}')

df4 = pd.DataFrame(results_exp4)
df4.to_csv('./sensitivity_output/tables/experiment_4_noise_levels.csv', index=False)
print('\nExperiment 4 complete!')
df4[['noise_level', 'stage_correlation', 'stage_mae', 'mean_confidence', 'success']]

In [ ]:
# Cell 13: Plot Experiment 4
df4_ok = df4[df4['success']]
if not df4_ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    noise_pct = df4_ok['noise_level'] * 100

    axes[0].plot(noise_pct, df4_ok['stage_correlation'], 'o-', lw=2, ms=8, color='purple')
    axes[0].set_xlabel('Noise Level (%)')
    axes[0].set_ylabel('Stage Correlation')
    axes[0].set_title('Robustness to Noise')
    axes[0].set_ylim([0, 1])
    axes[0].axhline(y=0.7, color='r', ls='--', alpha=0.5, label='Acceptable (0.7)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(noise_pct, df4_ok['stage_mae'], 'o-', lw=2, ms=8, color='red')
    axes[1].set_xlabel('Noise Level (%)')
    axes[1].set_ylabel('Stage MAE')
    axes[1].set_title('Stage Error vs Noise')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('./sensitivity_output/figures/exp4_noise_levels.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 14: Experiment 5 - Combined stress test
print('='*60)
print('EXPERIMENT 5: Combined Stress Test'.center(60))
print('='*60)

conditions = [
    (0.0,  0.0,  'Clean'),
    (0.1,  0.05, 'Mild'),
    (0.2,  0.1,  'Moderate'),
    (0.3,  0.15, 'Severe')
]

results_exp5 = []
for missing, noise, label in conditions:
    print(f'\nRunning {label} condition (missing={missing:.0%}, noise={noise:.0%})...')
    result = run_experiment(
        n_subjects=1000, n_biomarkers=13, n_scores=3,
        n_subtypes_true=3, n_subtypes_fit=3,
        missing_rate=missing, noise_level=noise, seed=42,
        use_gpu=USE_GPU
    )
    result['condition'] = label
    results_exp5.append(result)
    if result['success']:
        print(f'  Correlation: {result["stage_correlation"]:.3f} | Confidence: {result["mean_confidence"]:.3f}')

df5 = pd.DataFrame(results_exp5)
df5.to_csv('./sensitivity_output/tables/experiment_5_combined_stress.csv', index=False)
print('\nExperiment 5 complete!')
df5[['condition', 'missing_rate', 'noise_level', 'stage_correlation', 'mean_confidence', 'success']]

In [ ]:
# Cell 15: Plot Experiment 5
df5_ok = df5[df5['success']]
if not df5_ok.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(df5_ok))
    width = 0.25

    ax.bar(x - width, df5_ok['stage_correlation'], width, label='Stage Correlation', color='blue', alpha=0.8)
    ax.bar(x, df5_ok['mean_confidence'], width, label='Mean Confidence', color='green', alpha=0.8)
    max_mae = df5_ok['stage_mae'].max()
    if max_mae > 0:
        ax.bar(x + width, 1 - df5_ok['stage_mae']/max_mae, width, label='Normalized Accuracy', color='orange', alpha=0.8)

    ax.set_xlabel('Condition')
    ax.set_ylabel('Metric Value')
    ax.set_title('Combined Stress Test Results')
    ax.set_xticks(x)
    ax.set_xticklabels(df5_ok['condition'])
    ax.legend()
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('./sensitivity_output/figures/exp5_combined_stress.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 16: Summary of all experiments
print('='*70)
print('SENSITIVITY ANALYSIS SUMMARY'.center(70))
print('='*70)

all_dfs = {'Exp 1 (Subtypes)': df1, 'Exp 2 (Missing)': df2,
           'Exp 3 (Sample Size)': df3, 'Exp 4 (Noise)': df4,
           'Exp 5 (Combined)': df5}

for name, df in all_dfs.items():
    n_success = df['success'].sum()
    n_total = len(df)
    print(f'\n{name}: {n_success}/{n_total} successful')
    if n_success > 0:
        df_ok = df[df['success']]
        print(f'  Mean runtime: {df_ok["runtime"].mean():.1f}s')
        print(f'  Mean stage correlation: {df_ok["stage_correlation"].mean():.3f}')
        print(f'  Mean confidence: {df_ok["mean_confidence"].mean():.3f}')

print('\n' + '='*70)
print('All results saved to ./sensitivity_output/')
print('  figures/ - Publication-quality plots (300 DPI)')
print('  tables/  - CSV files with all metrics')

In [ ]:
# Cell 17: Download results
try:
    from google.colab import files
    import shutil
    shutil.make_archive('sensitivity_results', 'zip', './sensitivity_output')
    files.download('sensitivity_results.zip')
except ImportError:
    print('Not running on Colab. Results are in ./sensitivity_output/')